# Lab L3.2 · Behavioural customer segmentation with k-means and hierarchical clustering
**Horizon Gulf Bank · Project MIZAN** · CAIP AIP-210 · Certified AI Practitioner · Day 3 · ⏱ 45 min · **Level 2 · Guided starter (fill in the blanks) · ★ recommended level**

Segment 250,000 customers on behaviour, choose k with evidence, validate with a second algorithm and turn clusters into named personas.

### How this notebook works
- Every `✏️ BLANK n` marks code **you** write. Inline blanks look like `____`; block blanks say `# ...your code here...`.
- The markdown cell above each code cell explains **what** to write, **where**, and **why**.
- Stuck > 5 min? Open the hint in the HTML lab guide, or peek at the Level 3 solution.

### Scenario
**Horizon Gulf Bank (HGB)** is a fictional UAE retail bank with 250,000 customers. Its Board approved **Project MIZAN** — an AI decisioning platform covering credit, fraud, collateral valuation, cash logistics, customer retention and document automation. You are a junior ML practitioner in the MIZAN squad.

The Retail Marketing team runs one-size-fits-all campaigns. They want 4–7 actionable segments with clear personas so that product offers, channel and retention budget can be tailored. There is no label — this is **unsupervised** learning.

**Progressive architecture:** uses your `customers_clean.parquet`; segments become an enrichment feature and a monitoring dimension later.

In [ ]:
# ── Project MIZAN course setup ─ run this cell first ──────────────────────────
from pathlib import Path
import warnings, time
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate the course root (the folder that contains data/generate_data.py) from wherever this notebook lives
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "generate_data.py").exists())
DATA = ROOT / "data"          # raw synthetic datasets (read-only)
CKPT = ROOT / "checkpoints"   # hand-off artefacts between labs (lets you start any lab fresh)
CKPT.mkdir(exist_ok=True)
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.grid": True, "grid.alpha": .3})
RNG = 42
print("Course root:", ROOT)

## Step 1 · Select and transform behavioural features

**Why this matters.** Segments should describe **behaviour** the bank can act on — balances, engagement, product holding — not protected attributes. Money variables are log-transformed (otherwise a few millionaires form their own clusters), then everything is standardised so each feature contributes equally to distance.

#### ✏️ What to write in the next cell
- **Blank 1** *(inline)* — share of salary spent on card
- **Blank 2** *(inline)* — mean 0, std 1 for every feature

In [ ]:
from sklearn.preprocessing import StandardScaler
cust = pd.read_parquet(CKPT / "customers_clean.parquet")
B = pd.DataFrame({
    "log_balance": np.log1p(cust.avg_balance_aed), "log_salary": np.log1p(cust.monthly_salary_aed),
    "tenure_months": cust.tenure_months, "num_products": cust.num_products,
    "digital_logins_30d": cust.digital_logins_30d, "branch_visits_90d": cust.branch_visits_90d,
    "card_spend_ratio": ____,  # ✏️ B1
    "log_days_since_txn": np.log1p(cust.days_since_last_txn), "complaints_12m": cust.complaints_12m})
Xs = ____  # ✏️ B2
pd.DataFrame(Xs, columns=B.columns).agg(["mean", "std"]).round(2)

**✔ Checkpoint.** All means ≈ 0 and stds ≈ 1.

## Step 2 · Choose k with elbow and silhouette

**Why this matters.** `MiniBatchKMeans` fits on mini-batches, making 250k rows fast. Silhouette is O(n²), so we score it on a 15k-row sample. We look for a k where inertia’s improvement slows and silhouette is still reasonable — within the 4–7 range Marketing can operationalise.

#### ✏️ What to write in the next cell
- **Blank 3** *(inline)* — cluster separation on a sample

In [ ]:
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score
idx = np.random.default_rng(RNG).choice(len(Xs), 15_000, replace=False)
rows = []
for k in range(2, 10):
    km = MiniBatchKMeans(n_clusters=k, batch_size=4096, n_init=5, random_state=RNG).fit(Xs)
    rows.append((k, km.inertia_, ____))  # ✏️ B3
sel = pd.DataFrame(rows, columns=["k", "inertia", "silhouette"]).set_index("k")
sel.plot(subplots=True, layout=(1, 2), marker="o", figsize=(11, 3.8)); plt.show()
K = int(sel.loc[4:7, "silhouette"].idxmax())
print("chosen K =", K); sel.round(3)

**✔ Checkpoint.** Inertia falls smoothly; silhouette only ≈ 0.10–0.18 (typical for behavioural data). K = 4 is selected.

## Step 3 · Fit, profile and name the segments

**Why this matters.** Clusters are only valuable once a human can describe them. A **profile** in original units plus a z-score heat-map makes the personas obvious. Churn was *not* used for clustering, so churn by segment is an honest test of business relevance.

#### ✏️ What to write in the next cell
- **Blank 4** *(code block, ≈1 lines)* — name each segment from the heat-map: build PERSONA = {segment_id: "short persona name"} automatically from the strongest z-scores

In [ ]:
km = MiniBatchKMeans(n_clusters=K, batch_size=4096, n_init=10, random_state=RNG).fit(Xs)
cust["segment"] = km.labels_
profile = cust.assign(**B).groupby("segment").agg(share=("customer_id", lambda s: len(s) / len(cust)),
    salary=("monthly_salary_aed", "median"), balance=("avg_balance_aed", "median"), tenure=("tenure_months", "median"),
    products=("num_products", "mean"), logins=("digital_logins_30d", "mean"), branch=("branch_visits_90d", "mean"),
    days_idle=("days_since_last_txn", "median"), churn_rate=("churned_6m", "mean"))
display(profile.round(2))
centres = pd.DataFrame(km.cluster_centers_, columns=B.columns)
plt.figure(figsize=(11, 3.5)); plt.imshow(centres, cmap="RdBu_r", vmin=-1.5, vmax=1.5, aspect="auto")
plt.xticks(range(B.shape[1]), B.columns, rotation=35, ha="right"); plt.yticks(range(K)); plt.colorbar(label="z-score"); plt.title("Segment centres"); plt.show()
# ✏️ BLANK 4 — write ≈1 line(s): name each segment from the heat-map: build PERSONA = {segment_id: "short persona name"} automatically from the strongest z-scores
# ...your code here...
cust["persona"] = cust.segment.map(PERSONA); PERSONA

**✔ Checkpoint.** Recognisable groups emerge, e.g. *digital high-earners*, *dormant low-balance*, *new-to-bank*, *branch-loyal long-tenure*. Churn differs by 2–4× between segments.

## Step 4 · Cross-check with hierarchical clustering and PCA

**Why this matters.** If a different algorithm finds similar groups, the structure is real rather than an artefact of k-means. Ward linkage on a sample and the **adjusted Rand index** (0 = chance, 1 = identical) quantify the agreement. **PCA** gives a 2-D picture of the segments.

#### ✏️ What to write in the next cell
- **Blank 5** *(inline)* — agglomerative merge tree minimising within-cluster variance

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from sklearn.metrics import adjusted_rand_score
from sklearn.decomposition import PCA
h_idx = idx[:4000]
Z = ____  # ✏️ B5
plt.figure(figsize=(12, 3.5)); dendrogram(Z, truncate_mode="lastp", p=25); plt.title("Ward dendrogram (4k sample)"); plt.show()
h_lab = fcluster(Z, t=K, criterion="maxclust")
print("adjusted Rand index (Ward vs k-means):", round(adjusted_rand_score(km.labels_[h_idx], h_lab), 3))
pca = PCA(n_components=2, random_state=RNG).fit(Xs)
P2 = pca.transform(Xs[idx[:20000]])
plt.scatter(P2[:, 0], P2[:, 1], c=km.labels_[idx[:20000]], s=2, cmap="tab10")
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.title(f"Segments in PCA space ({pca.explained_variance_ratio_.sum():.0%} variance shown)"); plt.show()

**✔ Checkpoint.** ARI ≈ 0.3–0.6 (moderate agreement); PCA shows overlapping but distinct regions.

## Step 5 · Persist segments for downstream use

**Why this matters.** Segments become a feature for other models and a dimension for monitoring (e.g. fraud or churn rate by segment). We save the mapping and the fitted scaler + model so that new customers can be assigned consistently.

In [ ]:
import joblib
seg = cust[["customer_id", "segment", "persona"]]
assert seg.customer_id.is_unique and seg.segment.notna().all()
seg.to_parquet(CKPT / "customer_segments.parquet", index=False)
joblib.dump({"features": list(B.columns), "kmeans": km}, CKPT / "segmentation_model.joblib")
seg.segment.value_counts(normalize=True).sort_index().round(3)

**✔ Checkpoint.** Segment shares match the profile table; two files saved.

## ✅ Lab complete
Marketing has data-driven personas. **Next:** fraud detection with decision trees and random forests at 1.5M-row scale.